# Pools bronze layer

### Imports

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
# dbutils.widgets.text("entity_name", "")
# dbutils.widgets.text("load_pattern", "")

In [0]:
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Variables

In [0]:
query_variables = {
    "skip": 1000
}

### Execution

In [0]:
pool_query = """
query PoolsQuery($skip: Int!){
  pools(first: 1000,
    skip: $skip, 
    orderDirection: asc
  ) {
    id
    token0 { 
      symbol
      id
      decimals
    }
    token1 { 
      symbol
      id
      decimals
    }
    feeTier
    liquidity
    sqrtPrice
    tick
    createdAtTimestamp
    createdAtBlockNumber
  }
}
"""

In [0]:
response = requests.post(SUBGRAPH_URL, json={"query": pool_query, "variables": query_variables})

In [0]:
pools_df = spark.createDataFrame(response.json()["data"][entity_name])

pools_df = pools_df.withColumn("load_timestamp", current_timestamp())

In [0]:
if load_pattern == "1":
    print("*INFO*: Executing full load.")
    pools_df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "2":
    print("*INFO*: Executing incremental load.")
    pools_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "3":
    print("*INFO*: Executing differential load.")
    # DeltaTables merge
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")
else:
    print("*ERROR*: Invalid load pattern.")
    dbutils.notebook.exit("Invalid load pattern.")